In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

voice  = pd.read_csv('../results/results_voice.csv')
spiral = pd.read_csv('../results/results_spiral.csv')
mri    = pd.read_csv('../results/results_mri.csv')
modality_data = {'Voice': voice, 'Spiral': spiral, 'MRI': mri}
mods = ['Voice', 'Spiral', 'MRI']

N_REPEATS = 300
RNG = np.random.default_rng(42)

def evaluate_weights(weights, n_repeats=N_REPEATS):
    per_repeat_bal_acc, per_repeat_auc = [], []
    for _ in range(n_repeats):
        fused_true, fused_prob = [], []
        for label in [0, 1]:
            pools = [modality_data[m][modality_data[m]['label'] == label]['prob'].values for m in mods]
            n = min(len(p) for p in pools)
            idxs = [RNG.choice(len(p), size=n, replace=False) for p in pools]
            probs_matrix = np.stack([pools[i][idxs[i]] for i in range(len(mods))], axis=1)
            w = np.array([weights[m] for m in mods])
            fused = probs_matrix @ w
            fused_true.extend([label] * n)
            fused_prob.extend(fused)
        fused_true, fused_prob = np.array(fused_true), np.array(fused_prob)
        fused_pred = (fused_prob > 0.5).astype(int)
        per_repeat_bal_acc.append(balanced_accuracy_score(fused_true, fused_pred))
        per_repeat_auc.append(roc_auc_score(fused_true, fused_prob))
    return np.mean(per_repeat_bal_acc), np.mean(per_repeat_auc)

step = 0.1
grid = []
for wv in np.arange(0, 1.0001, step):
    for ws in np.arange(0, 1.0001 - wv, step):
        wm = round(1 - wv - ws, 2)
        if wm < -1e-9:
            continue
        grid.append({'Voice': round(wv, 2), 'Spiral': round(ws, 2), 'MRI': max(wm, 0)})

print(f"Всего комбинаций весов: {len(grid)}")

results = []
for w in grid:
    bal_acc, auc = evaluate_weights(w)
    results.append({**w, 'balanced_acc': bal_acc, 'auc': auc})

results_df = pd.DataFrame(results).sort_values('balanced_acc', ascending=False)

print("\n=== Топ-10 по balanced accuracy ===")
print(results_df.head(10).to_string(index=False))

print("\n=== Топ-10 по AUC ===")
print(results_df.sort_values('auc', ascending=False).head(10).to_string(index=False))

equal_w = {'Voice': round(1/3, 2), 'Spiral': round(1/3, 2), 'MRI': round(1/3, 2)}
original_w = {'Voice': 0.2, 'Spiral': 0.3, 'MRI': 0.5}

eq_bal, eq_auc = evaluate_weights(equal_w)
orig_bal, orig_auc = evaluate_weights(original_w)

print(f"\nРавные веса        (0.33/0.33/0.33): balanced_acc={eq_bal:.3f}, auc={eq_auc:.3f}")
print(f"Оригинальные веса  (0.2/0.3/0.5):     balanced_acc={orig_bal:.3f}, auc={orig_auc:.3f}")

best = results_df.iloc[0]
print(f"\nЛучшая по balanced_acc: Voice={best['Voice']}, Spiral={best['Spiral']}, MRI={best['MRI']} "
      f"→ balanced_acc={best['balanced_acc']:.3f}, auc={best['auc']:.3f}")

results_df.to_csv('weight_sensitivity_table.csv', index=False)
print("\nСохранено: weight_sensitivity_table.csv")

Всего комбинаций весов: 66

=== Топ-10 по balanced accuracy ===
 Voice  Spiral  MRI  balanced_acc      auc
   0.1     0.8  0.1      0.906208 0.897472
   0.1     0.9  0.0      0.905125 0.897222
   0.1     0.7  0.2      0.901889 0.894778
   0.1     0.6  0.3      0.898597 0.899139
   0.2     0.8  0.0      0.880347 0.897167
   0.2     0.7  0.1      0.878681 0.902611
   0.1     0.5  0.4      0.876917 0.894972
   0.2     0.6  0.2      0.869236 0.906306
   0.0     0.8  0.2      0.868153 0.894861
   0.0     0.6  0.4      0.866389 0.885778

=== Топ-10 по AUC ===
 Voice  Spiral  MRI  balanced_acc      auc
   0.3     0.5  0.2      0.838153 0.922278
   0.3     0.4  0.3      0.790542 0.919556
   0.4     0.4  0.2      0.759083 0.918694
   0.3     0.3  0.4      0.769625 0.917694
   0.5     0.4  0.1      0.755042 0.917667
   0.4     0.6 -0.0      0.799097 0.917528
   0.4     0.5  0.1      0.772111 0.917250
   0.4     0.3  0.3      0.756944 0.916833
   0.3     0.6  0.1      0.854458 0.916833
   0.5    